In [1]:
# 04_simulation_verification.ipynb
# Discrete-event verification of the coupled delegation -> M/G/c review queue.
# Checks: (1) M/G/1 exact P-K reproduced to <0.1%; (2) M/G/c KLB residual gap;
# (3) bottleneck migration (pre-AI vs post-AI review load).

## Coupled simulation and analytical cross-check

In [2]:
# Coupled simulation v3: M/G/1 exact (P-K) + M/G/c (KLB approx) verification
import numpy as np, math, json, csv
tasks=[]
with open("../data/tasks.csv") as fh:
    for r in csv.DictReader(fh): tasks.append(r)
tau={t["task_id"]:float(t["pre_ai_hours"]) for t in tasks}
vv={t["task_id"]:float(t["verification_hours"]) for t in tasks}
ctype={t["task_id"]:t["compression_type"] for t in tasks}
TIDS=list(tau.keys())
qp=json.load(open("../data/queue_params.json"))
p_prime=qp["client_mix"]["prime_share"]

def stage_mean(x): return sum((1-x[t])*tau[t]+x[t]*vv[t] for t in TIDS)
def make_two_point(mean,cs=0.6,p=p_prime):
    spread=mean*math.sqrt((cs**2)/(p*(1-p)))
    return max(mean-(1-p)*spread,0.05), mean+p*spread

def pk_wq_mg1(lam,ES,ES2):  # exact Pollaczek-Khinchine, single server
    rho=lam*ES
    return lam*ES2/(2*(1-rho))
def erlang_c_wq(lam,mu,c):
    a=lam/mu; rho=a/c
    if rho>=1: return float('inf')
    s=sum(a**n/math.factorial(n) for n in range(c))
    last=a**c/(math.factorial(c)*(1-rho)); p0=1/(s+last)
    return last*p0/(c*mu-lam)
def klb_wq(lam,ES,cs2,c): return erlang_c_wq(lam,1/ES,c)*(1+cs2)/2
def sim(lam,sf,ss,p,c,n=12000,w=1000,seed=0):
    rng=np.random.default_rng(seed); free=[0.0]*c; t=0.0; W=[]
    for i in range(n+w):
        t+=rng.exponential(1/lam); svc=sf if rng.random()<p else ss
        j=int(np.argmin(free)); st=max(t,free[j]); free[j]=st+svc
        if i>=w: W.append(st-t)
    return float(np.mean(W))

x_base={t:(0.9 if ctype[t]=="high_compression" else (0.5 if ctype[t]=="partial" else 0.0)) for t in TIDS}
ES=stage_mean(x_base); cs=0.6; sf,ss=make_two_point(ES,cs); cs2=cs**2
ES2=p_prime*sf**2+(1-p_prime)*ss**2  # exact 2nd moment of the two-point mixture

print("="*72); print("EXP 1 - M/G/1 EXACT (Pollaczek-Khinchine) vs simulation, c=1"); print("="*72)
print(f"E[S]={ES:.3f}h  E[S^2]={ES2:.3f}  CS^2={cs2:.3f}  s_fast={sf:.3f}  s_slow={ss:.3f}")
mu=1/ES
rows1=[]
for rho in [0.4,0.6,0.8]:
    lam=rho*mu
    wk=pk_wq_mg1(lam,ES,ES2)
    S=[sim(lam,sf,ss,p_prime,1,seed=s) for s in range(30)]
    wsim=np.mean(S); e=100*(wsim-wk)/wk
    rows1.append((rho,lam,wk,wsim,e))
    print(f"  rho={rho:.1f} lambda={lam:.4f}  PK Wq={wk:.4f}h  sim={wsim:.4f}h  err={e:+.2f}%")

print(); print("="*72); print("EXP 2 - M/G/c (KLB approximation) vs simulation, c=12"); print("="*72)
c=12
rows2=[]
for rho in [0.4,0.6,0.8]:
    lam=rho*c*mu
    wk=klb_wq(lam,ES,cs2,c)
    S=[sim(lam,sf,ss,p_prime,c,seed=s) for s in range(30)]
    wsim=np.mean(S); e=100*(wsim-wk)/wk
    rows2.append((rho,lam,wk,wsim,e))
    print(f"  rho={rho:.1f} lambda={lam:.4f}  KLB Wq={wk:.4f}h  sim={wsim:.4f}h  err={e:+.2f}%")
print("  (KLB is an approximation; positive error at low rho is a known KLB property.)")

print(); print("="*72); print("EXP 3 - Bottleneck migration (pre-AI vs post-AI)"); print("="*72)
review_pre=sum(tau[t] for t in TIDS); review_post=stage_mean(x_base); review_full=sum(vv[t] for t in TIDS)
add=sum(vv[t]-tau[t] for t in ['T3','T6'])
print(f"  Pre-AI human review load/case  : {review_pre:.2f} h (distributed human pipeline)")
print(f"  Post-AI baseline delegation    : {review_post:.2f} h (concentrated at review)")
print(f"  Naive full delegation          : {review_full:.2f} h")
print(f"  Delegating Becker tasks adds   : {add:.1f} h -> optimum keeps T3,T6 human")

out={"ES":ES,"ES2":ES2,"cs2":cs2,"s_fast":sf,"s_slow":ss,"c":c,
 "mg1_exact":[{"rho":r[0],"lambda":r[1],"wq_pk":r[2],"wq_sim":r[3],"err_pct":r[4]} for r in rows1],
 "mgc_klb":[{"rho":r[0],"lambda":r[1],"wq_klb":r[2],"wq_sim":r[3],"err_pct":r[4]} for r in rows2],
 "migration":{"review_preAI":review_pre,"review_postAI":review_post,"review_full":review_full,"add_becker":add}}
json.dump(out,open("../results/tables/sim_results.json","w"),indent=2)
print("\nsaved sim_results.json")


EXP 1 - M/G/1 EXACT (Pollaczek-Khinchine) vs simulation, c=1
E[S]=4.090h  E[S^2]=22.750  CS^2=0.360  s_fast=2.086  s_slow=7.096


  rho=0.4 lambda=0.0978  PK Wq=1.8541h  sim=1.8532h  err=-0.05%


  rho=0.6 lambda=0.1467  PK Wq=4.1718h  sim=4.1692h  err=-0.06%


  rho=0.8 lambda=0.1956  PK Wq=11.1248h  sim=11.1321h  err=+0.07%

EXP 2 - M/G/c (KLB approximation) vs simulation, c=12


  rho=0.4 lambda=1.1736  KLB Wq=0.0017h  sim=0.0020h  err=+23.76%


  rho=0.6 lambda=1.7604  KLB Wq=0.0433h  sim=0.0494h  err=+14.23%


  rho=0.8 lambda=2.3472  KLB Wq=0.4274h  sim=0.4533h  err=+6.04%
  (KLB is an approximation; positive error at low rho is a known KLB property.)

EXP 3 - Bottleneck migration (pre-AI vs post-AI)
  Pre-AI human review load/case  : 12.00 h (distributed human pipeline)
  Post-AI baseline delegation    : 4.09 h (concentrated at review)
  Naive full delegation          : 4.90 h
  Delegating Becker tasks adds   : 2.0 h -> optimum keeps T3,T6 human

saved sim_results.json


## Verification figure (grayscale, 600 dpi, PNG + PDF)

In [3]:
# Simulation verification figure (grayscale, 600 dpi, png+pdf, no caption)
import json, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
r=json.load(open("../results/tables/sim_results.json"))
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi":600,"font.size":11,"axes.edgecolor":"0.2","grid.color":"0.85"})

fig,ax=plt.subplots(1,2,figsize=(9,3.8))
# left: M/G/1 exact
d1=r["mg1_exact"]
rho=[x["rho"] for x in d1]; pk=[x["wq_pk"] for x in d1]; sm=[x["wq_sim"] for x in d1]
ax[0].plot(rho,pk,"o-",color="0.15",lw=1.4,label="P-K analytical",markersize=7)
ax[0].plot(rho,sm,"s--",color="0.55",lw=1.4,label="Simulation",markersize=6,markerfacecolor="white")
ax[0].set_xlabel(r"Traffic intensity $\rho$"); ax[0].set_ylabel(r"Mean waiting time $W_q$ (h)")
ax[0].set_title("M/G/1 (exact, $c=1$)",fontsize=11); ax[0].legend(frameon=True,edgecolor="0.5")
# right: M/G/c KLB
d2=r["mgc_klb"]
rho2=[x["rho"] for x in d2]; kl=[x["wq_klb"] for x in d2]; sm2=[x["wq_sim"] for x in d2]
ax[1].plot(rho2,kl,"o-",color="0.15",lw=1.4,label="KLB approximation",markersize=7)
ax[1].plot(rho2,sm2,"s--",color="0.55",lw=1.4,label="Simulation",markersize=6,markerfacecolor="white")
ax[1].set_xlabel(r"Traffic intensity $\rho$"); ax[1].set_ylabel(r"Mean waiting time $W_q$ (h)")
ax[1].set_title("M/G/c (KLB, $c=12$)",fontsize=11); ax[1].legend(frameon=True,edgecolor="0.5")
ax[1].set_yscale("log")
plt.tight_layout()
for ext in ("png","pdf"):
    fig.savefig(f"../results/figures/fig6_sim_verification.{ext}",dpi=600,bbox_inches="tight")
    # also into repo
    fig.savefig(f"/home/claude/hitl_ai_repo/results/../results/figures/fig6_sim_verification.{ext}",dpi=600,bbox_inches="tight")
plt.close(fig)
print("fig6 saved")


fig6 saved
